# Extended Lab: Conditionals (`CASE`) & NULL Handling in SQL

**Goal:** Strengthen your ability to classify rows, handle missing data, and combine both techniques in realistic business queries.

This lab continues from the optional practice on conditionals and NULL values.  
You will work with the same **Chinook** database (`employees` and `customers` tables).

---

### Quick Schema Reminder

**employees**  
`EmployeeId`, `FirstName`, `LastName`, `Title`, `ReportsTo`, `BirthDate`, `HireDate`, `City`, `State`, `Country`, `Email`, ...

**customers**  
`CustomerId`, `FirstName`, `LastName`, `Company`, `City`, `State`, `Country`, `Phone`, `Fax`, `Email`, `SupportRepId`, ...

---

### How to use this notebook
1. Read the business scenario.
2. Write the query in the empty code cell.
3. Compare your result with the expected output.
4. Use the hints only if you get stuck.
5. Solutions are at the bottom (try not to peek too early!).

## Exercise 1 – Review (Warm-up)

Classify each employee into an **Area**:
- Title contains `"IT"` → `"IT"`
- Title contains `"Sales"` → `"Sales"`
- Everything else → `"Management"`

Return: `EmployeeId`, `FirstName`, `LastName`, `Area`

**Expected output (first rows):**
```
+------------+-----------+----------+------------+
| EmployeeId | FirstName | LastName | Area       |
+------------+-----------+----------+------------+
| 1          | Andrew    | Adams    | Management |
| 2          | Nancy     | Edwards  | Sales      |
| 3          | Jane      | Peacock  | Sales      |
| 4          | Margaret  | Park     | Sales      |
| 5          | Steve     | Johnson  | Sales      |
| 6          | Michael   | Mitchell | IT         |
| 7          | Robert    | King     | IT         |
| 8          | Laura     | Callahan | IT         |
+------------+-----------+----------+------------+
```

## Exercise 2 – Review (Warm-up)

Find all **independent customers** (those with no company information).

Return: `CustomerId`, `FirstName`, `LastName`  
Filter: `Company IS NULL`

## Exercise 3 – More Precise Classification

The HR team refined the rules:

- If `Title` **starts with** `"IT"` → `"IT"`
- If `Title` **starts with** `"Sales"` → `"Sales"`
- Otherwise → `"Management"`

> Tip: Use `LIKE 'IT%'` and `LIKE 'Sales%'` (no leading `%`).

Return the same columns as Exercise 1.

**Expected output:** Same 8 rows as Exercise 1 (the data happens to match both patterns).

## Exercise 4 – Hierarchical Level with CASE

Create a new column called `Level`:

- `Title` contains `"Manager"` → `"Manager"`
- `Title` contains `"Staff"` or `"Agent"` → `"Individual Contributor"`
- Otherwise → `"Executive"`

Also keep the original `Title`.

Return: `EmployeeId`, `FirstName`, `LastName`, `Title`, `Level`

**Expected output:**
```
+------------+-----------+----------+----------------------+------------------------+
| EmployeeId | FirstName | LastName | Title                | Level                  |
+------------+-----------+----------+----------------------+------------------------+
| 1          | Andrew    | Adams    | General Manager      | Manager                |
| 2          | Nancy     | Edwards  | Sales Manager        | Manager                |
| 3          | Jane      | Peacock  | Sales Support Agent  | Individual Contributor |
| 4          | Margaret  | Park     | Sales Support Agent  | Individual Contributor |
| 5          | Steve     | Johnson  | Sales Support Agent  | Individual Contributor |
| 6          | Michael   | Mitchell | IT Manager           | Manager                |
| 7          | Robert    | King     | IT Staff             | Individual Contributor |
| 8          | Laura     | Callahan | IT Staff             | Individual Contributor |
+------------+-----------+----------+----------------------+------------------------+
```

## Exercise 5 – Handling NULLs with COALESCE

Customers sometimes have missing phone or fax numbers.

Create a clean contact column called `PrimaryContact`:

- Prefer `Phone`
- If `Phone` is NULL, use `Fax`
- If both are NULL, show `'No contact info'`

Return: `CustomerId`, `FirstName`, `LastName`, `Phone`, `Fax`, `PrimaryContact`  
Limit the result to the first 15 rows for readability.

**Hints:**
- `COALESCE(Phone, Fax, 'No contact info')` is the cleanest way.
- You can also do it with nested `CASE` statements.

## Exercise 6 – Company Type Classification

Classify customers into a `CustomerType`:

- `Company IS NOT NULL` → `'Corporate'`
- `Company IS NULL` → `'Individual'`

Return: `CustomerId`, `FirstName`, `LastName`, `Company`, `CustomerType`  
Order by `CustomerType`, then `LastName`.

Show only the first 20 rows.

## Exercise 7 – Combined CASE + NULL Logic

The marketing team wants a more detailed segment:

- If `Company` is not null **and** `Country = 'USA'` → `'US Corporate'`
- If `Company` is not null **and** `Country != 'USA'` → `'International Corporate'`
- If `Company` is null **and** `Country = 'USA'` → `'US Individual'`
- Otherwise → `'International Individual'`

Return: `CustomerId`, `FirstName`, `LastName`, `Company`, `Country`, `Segment`  
Order by `Segment`, `LastName`.

Limit to 25 rows.

**Hints:**
- You can write multiple `WHEN` conditions that combine `IS NULL` / `IS NOT NULL` with country checks.
- Order of the `WHEN` clauses matters (most specific first).

## Exercise 8 – Manager Status + NULL ReportsTo

In the `employees` table, the top manager has `ReportsTo = NULL`.

Create two new columns:

1. `HasManager` → `'Yes'` if `ReportsTo` is not null, `'No'` otherwise
2. `ReportsToDisplay` → show the actual `ReportsTo` value, or `'Top Level'` when it is NULL

Return: `EmployeeId`, `FirstName`, `LastName`, `Title`, `ReportsTo`, `HasManager`, `ReportsToDisplay`

**Expected output:**
```
+------------+-----------+----------+----------------------+-----------+------------+------------------+
| EmployeeId | FirstName | LastName | Title                | ReportsTo | HasManager | ReportsToDisplay |
+------------+-----------+----------+----------------------+-----------+------------+------------------+
| 1          | Andrew    | Adams    | General Manager      | NULL      | No         | Top Level        |
| 2          | Nancy     | Edwards  | Sales Manager        | 1         | Yes        | 1                |
| 3          | Jane      | Peacock  | Sales Support Agent  | 2         | Yes        | 2                |
| 4          | Margaret  | Park     | Sales Support Agent  | 2         | Yes        | 2                |
| 5          | Steve     | Johnson  | Sales Support Agent  | 2         | Yes        | 2                |
| 6          | Michael   | Mitchell | IT Manager           | 1         | Yes        | 1                |
| 7          | Robert    | King     | IT Staff             | 6         | Yes        | 6                |
| 8          | Laura     | Callahan | IT Staff             | 6         | Yes        | 6                |
+------------+-----------+----------+----------------------+-----------+------------+------------------+
```

## Exercise 9 – Challenge: Full Customer Profile Flag

Create a completeness flag called `ProfileStatus`:

- If `Company`, `Phone`, **and** `Fax` are all present → `'Complete'`
- If only `Company` is missing but both phone & fax exist → `'Missing Company'`
- If `Company` exists but either phone or fax is missing → `'Missing Contact'`
- Otherwise → `'Incomplete'`

Return: `CustomerId`, `FirstName`, `LastName`, `Company`, `Phone`, `Fax`, `ProfileStatus`  
Order by `ProfileStatus`, `CustomerId`.

Limit to 30 rows.

## Exercise 10 – Bonus Challenge (Optional)

Combine everything you learned.

Write a single query that returns for **every employee**:

- `EmployeeId`, `FirstName`, `LastName`, `Title`
- `Area` (IT / Sales / Management – same rules as Exercise 1)
- `Level` (Manager / Individual Contributor / Executive – same rules as Exercise 4)
- `HasManager` (`'Yes'` / `'No'`)
- `ReportsToName` → the first name of the manager they report to, or `'None'` if they are the top manager

> You will need a self-join (or a correlated subquery) for the manager’s name.

Order the result by `Area`, then `Level`, then `LastName`.

---

# SOLUTIONS

Try the exercises first. Solutions are provided below for self-checking.

### Solution – Exercise 1 & 3

```sql
SELECT 
    EmployeeId, 
    FirstName, 
    LastName,
    CASE 
        WHEN Title LIKE '%IT%' THEN 'IT'
        WHEN Title LIKE '%Sales%' THEN 'Sales'
        ELSE 'Management'
    END AS Area
FROM employees;
```

(For the stricter “starts with” version replace with `LIKE 'IT%'` and `LIKE 'Sales%'`.)

### Solution – Exercise 2

```sql
SELECT CustomerId, FirstName, LastName
FROM customers
WHERE Company IS NULL;
```

### Solution – Exercise 4

```sql
SELECT 
    EmployeeId,
    FirstName,
    LastName,
    Title,
    CASE 
        WHEN Title LIKE '%Manager%' THEN 'Manager'
        WHEN Title LIKE '%Staff%' OR Title LIKE '%Agent%' THEN 'Individual Contributor'
        ELSE 'Executive'
    END AS Level
FROM employees;
```

### Solution – Exercise 5

```sql
SELECT 
    CustomerId,
    FirstName,
    LastName,
    Phone,
    Fax,
    COALESCE(Phone, Fax, 'No contact info') AS PrimaryContact
FROM customers
LIMIT 15;
```

Equivalent with CASE:

```sql
SELECT 
    CustomerId,
    FirstName,
    LastName,
    Phone,
    Fax,
    CASE 
        WHEN Phone IS NOT NULL THEN Phone
        WHEN Fax IS NOT NULL THEN Fax
        ELSE 'No contact info'
    END AS PrimaryContact
FROM customers
LIMIT 15;
```

### Solution – Exercise 6

```sql
SELECT 
    CustomerId,
    FirstName,
    LastName,
    Company,
    CASE 
        WHEN Company IS NOT NULL THEN 'Corporate'
        ELSE 'Individual'
    END AS CustomerType
FROM customers
ORDER BY CustomerType, LastName
LIMIT 20;
```

### Solution – Exercise 7

```sql
SELECT 
    CustomerId,
    FirstName,
    LastName,
    Company,
    Country,
    CASE 
        WHEN Company IS NOT NULL AND Country = 'USA' THEN 'US Corporate'
        WHEN Company IS NOT NULL AND Country != 'USA' THEN 'International Corporate'
        WHEN Company IS NULL AND Country = 'USA' THEN 'US Individual'
        ELSE 'International Individual'
    END AS Segment
FROM customers
ORDER BY Segment, LastName
LIMIT 25;
```

### Solution – Exercise 8

```sql
SELECT 
    EmployeeId,
    FirstName,
    LastName,
    Title,
    ReportsTo,
    CASE 
        WHEN ReportsTo IS NOT NULL THEN 'Yes'
        ELSE 'No'
    END AS HasManager,
    COALESCE(CAST(ReportsTo AS TEXT), 'Top Level') AS ReportsToDisplay
FROM employees;
```

### Solution – Exercise 9

```sql
SELECT 
    CustomerId,
    FirstName,
    LastName,
    Company,
    Phone,
    Fax,
    CASE 
        WHEN Company IS NOT NULL AND Phone IS NOT NULL AND Fax IS NOT NULL THEN 'Complete'
        WHEN Company IS NULL AND Phone IS NOT NULL AND Fax IS NOT NULL THEN 'Missing Company'
        WHEN Company IS NOT NULL AND (Phone IS NULL OR Fax IS NULL) THEN 'Missing Contact'
        ELSE 'Incomplete'
    END AS ProfileStatus
FROM customers
ORDER BY ProfileStatus, CustomerId
LIMIT 30;
```

### Solution – Exercise 10 (Bonus)

```sql
SELECT 
    e.EmployeeId,
    e.FirstName,
    e.LastName,
    e.Title,
    CASE 
        WHEN e.Title LIKE '%IT%' THEN 'IT'
        WHEN e.Title LIKE '%Sales%' THEN 'Sales'
        ELSE 'Management'
    END AS Area,
    CASE 
        WHEN e.Title LIKE '%Manager%' THEN 'Manager'
        WHEN e.Title LIKE '%Staff%' OR e.Title LIKE '%Agent%' THEN 'Individual Contributor'
        ELSE 'Executive'
    END AS Level,
    CASE 
        WHEN e.ReportsTo IS NOT NULL THEN 'Yes'
        ELSE 'No'
    END AS HasManager,
    COALESCE(m.FirstName, 'None') AS ReportsToName
FROM employees e
LEFT JOIN employees m ON e.ReportsTo = m.EmployeeId
ORDER BY Area, Level, e.LastName;
```

---

### Key Takeaways

| Concept | Pattern |
|---------|---------|
| Simple classification | `CASE WHEN ... THEN ... ELSE ... END` |
| Starts-with match | `LIKE 'value%'` |
| Contains match | `LIKE '%value%'` |
| NULL check | `IS NULL` / `IS NOT NULL` |
| Fallback value | `COALESCE(col1, col2, 'default')` |
| Multiple conditions | Combine with `AND` / `OR` inside `WHEN` |
| Self-join for hierarchy | `LEFT JOIN employees m ON e.ReportsTo = m.EmployeeId` |

Practice these patterns until they feel natural — they appear constantly in real data work (customer segmentation, employee reporting, data quality checks, etc.).